### Where Qwen3.5-4B fails: accuracy vs. clip duration

Every take is shown to the model as 16 frames sampled across the whole clip, so the longer the take, the
sparser the frames. This compares the midterm evaluation results (`eval_*_test*.csv` from
`idea_1_midterm.ipynb`; zero-shot and LoRA checkpoint 50, each with video, video + commentary, and commentary
only) with the duration of each test take. Duration is largely set by the scenario (a few seconds of
bouldering vs. many minutes of cooking or music), so besides raw duration quartiles, durations are compared
within each scenario.

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

import av
import numpy as np

DATA_ROOT = next(
    p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "annotations").is_dir()
)
NUM_FRAMES = 16  # frames sampled per take, as in training and evaluation

# Midterm evaluation results, one CSV per model x input (see evaluate() in idea_1_midterm.ipynb).
RUNS = {
    ("Zero-shot", "Video"): "eval_base_test_video_only.csv",
    ("Zero-shot", "Video + commentary"): "eval_base_test.csv",
    ("Zero-shot", "Commentary"): "eval_base_test_commentary_only.csv",
    ("Fine-tuned", "Video"): "eval_checkpoint-50_test_video_only.csv",
    ("Fine-tuned", "Video + commentary"): "eval_checkpoint-50_test.csv",
    ("Fine-tuned", "Commentary"): "eval_checkpoint-50_test_commentary_only.csv",
}
runs = {}
for key, path in RUNS.items():
    with open(path, newline="", encoding="utf-8") as file:
        runs[key] = {row["take_name"]: row for row in csv.DictReader(file)}
take_names = list(runs["Fine-tuned", "Video"])
assert all(set(rows) == set(take_names) for rows in runs.values()), "the runs should score the same takes"

# The ego video each take was scored on (only the downscaled 448p copies are on disk).
with open(DATA_ROOT / "annotations" / "proficiency_demonstrator_val.json", encoding="utf-8") as file:
    val_annotations = json.load(file)["annotations"]
video_paths = {}
for m in val_annotations:
    parts = m["video_paths"]["ego"].split("/")
    parts[-1:-1] = ["downscaled", "448"]
    video_paths[parts[1]] = DATA_ROOT.joinpath(*parts)

def duration(path):
    """Length of the video in seconds."""
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        if stream.duration is not None:
            return float(stream.duration * stream.time_base)
        return container.duration / av.time_base

reference = runs["Fine-tuned", "Video"]
durations = np.array([duration(video_paths[t]) for t in take_names])
scenarios = np.array([reference[t]["scenario"] for t in take_names])
true = np.array([int(reference[t]["true"]) for t in take_names])
correct = {key: np.array([int(rows[t]["pred"]) == int(rows[t]["true"]) for t in take_names]) for key, rows in runs.items()}
# The model's expected score under its own probabilities, a graded version of its prediction.
expected_score = {key: np.array([sum(s * float(rows[t][f"p{s}"]) for s in range(1, 5)) for t in take_names])
                  for key, rows in runs.items()}

print(f"{len(take_names)} test takes, {durations.min():.0f}-{durations.max():.0f} s (median {np.median(durations):.0f} s)\n")
print(f"{'scenario':<15}{'n':>4}{'median s':>10}{'range s':>14}{'s between frames':>18}")
for scenario in np.unique(scenarios):
    d = durations[scenarios == scenario]
    print(f"{scenario:<15}{len(d):>4}{np.median(d):>10.0f}{f'{d.min():.0f}-{d.max():.0f}':>14}{np.median(d) / NUM_FRAMES:>18.1f}")

164 test takes, 8-852 s (median 57 s)

scenario          n  median s       range s  s between frames
Basketball       42        55        11-168               3.4
Cooking          12       288       133-637              18.0
Dance            38        60        48-237               3.8
Music            16       274       111-852              17.1
Rock Climbing    51        27         8-100               1.7
Soccer            5       156       109-222               9.7


In [2]:
# Raw duration quartiles. These mostly separate scenarios, so each bin's scenario mix is listed.
edges = np.quantile(durations, [0, 0.25, 0.5, 0.75, 1])
quartile = np.searchsorted(edges[1:-1], durations, side="right")
quartile_labels = []
for q in range(4):
    in_bin = quartile == q
    top = ", ".join(f"{s} {n}" for s, n in Counter(scenarios[in_bin]).most_common(2))
    quartile_labels.append(f"{durations[in_bin].min():.0f}-{durations[in_bin].max():.0f} s")
    print(f"Q{q + 1}: {quartile_labels[-1]:>12} ({in_bin.sum()} takes, {np.median(durations[in_bin]) / NUM_FRAMES:.1f} s between frames; "
          f"mostly {top})")

print(f"\n{'model':<12}{'input':<20}" + "".join(f"{f'Q{q + 1}':>8}" for q in range(4)) + f"{'all':>8}")
for (model_name, input_name), ok in correct.items():
    print(f"{model_name:<12}{input_name:<20}" + "".join(f"{ok[quartile == q].mean():>8.3f}" for q in range(4)) + f"{ok.mean():>8.3f}")
print(f"{'majority class within bin':<32}" + "".join(
    f"{np.bincount(true[quartile == q]).max() / (quartile == q).sum():>8.3f}" for q in range(4)))

Q1:       8-36 s (41 takes, 1.4 s between frames; mostly Rock Climbing 33, Basketball 8)
Q2:      36-57 s (41 takes, 3.1 s between frames; mostly Basketball 16, Dance 16)
Q3:     57-140 s (41 takes, 4.2 s between frames; mostly Basketball 17, Dance 11)
Q4:    141-852 s (41 takes, 14.2 s between frames; mostly Music 15, Dance 11)

model       input                     Q1      Q2      Q3      Q4     all
Zero-shot   Video                  0.341   0.122   0.244   0.317   0.256
Zero-shot   Video + commentary     0.317   0.366   0.415   0.463   0.390
Zero-shot   Commentary             0.171   0.341   0.439   0.439   0.348
Fine-tuned  Video                  0.390   0.268   0.195   0.341   0.299
Fine-tuned  Video + commentary     0.293   0.366   0.439   0.512   0.402
Fine-tuned  Commentary             0.293   0.341   0.463   0.585   0.421
majority class within bin          0.463   0.512   0.390   0.341


In [3]:
# Within scenario: each scenario's takes split at that scenario's median duration, so both halves have the
# same scenario mix. The p-value tests the association between log duration and correctness (both centered
# per scenario) against 10,000 shuffles of the durations within each scenario.
rng = np.random.default_rng(0)
groups = [np.flatnonzero(scenarios == s) for s in np.unique(scenarios)]
log_duration = np.log(durations)

def within_scenario_corr(x, y):
    """Pearson correlation of x and y after subtracting each scenario's mean from both."""
    x, y = x.astype(float).copy(), y.astype(float).copy()
    for idx in groups:
        x[idx] -= x[idx].mean()
        y[idx] -= y[idx].mean()
    return np.corrcoef(x, y)[0, 1]

def permutation_test(x, y, n=10_000):
    """Within-scenario correlation of x and y and its two-sided p-value (x shuffled within each scenario)."""
    observed = within_scenario_corr(x, y)
    null = np.empty(n)
    for k in range(n):
        shuffled = x.copy()
        for idx in groups:
            shuffled[idx] = x[rng.permutation(idx)]
        null[k] = within_scenario_corr(shuffled, y)
    return observed, (np.abs(null) >= abs(observed)).mean()

def p_text(p, n=10_000):
    return f"{p:.3f}" if p > 0 else f"<{1 / n:g}"

longer = np.zeros(len(durations), dtype=bool)
for idx in groups:
    longer[idx] = durations[idx] > np.median(durations[idx])

print(f"Shorter half: {(~longer).sum()} takes, longer half: {longer.sum()} takes\n")
print(f"{'model':<12}{'input':<20}{'shorter half':>14}{'longer half':>13}{'r(log dur., correct)':>22}{'p':>8}")
for (model_name, input_name), ok in correct.items():
    r, p = permutation_test(log_duration, ok)
    print(f"{model_name:<12}{input_name:<20}{ok[~longer].mean():>14.3f}{ok[longer].mean():>13.3f}{r:>22.3f}{p_text(p):>8}")

Shorter half: 83 takes, longer half: 81 takes

model       input                 shorter half  longer half  r(log dur., correct)       p


Zero-shot   Video                        0.241        0.272                 0.096   0.236


Zero-shot   Video + commentary           0.337        0.444                 0.120   0.131


Zero-shot   Commentary                   0.277        0.420                 0.115   0.140


Fine-tuned  Video                        0.265        0.333                 0.148   0.065


Fine-tuned  Video + commentary           0.398        0.407                 0.116   0.141


Fine-tuned  Commentary                   0.386        0.457                 0.127   0.114


In [4]:
# Is duration itself a cue? Within each scenario, does it relate to the true level (e.g. stronger climbers
# finishing faster), and does the model's expected score follow it? The video prompt carries frame
# timestamps, so the model could read duration directly.
r, p = permutation_test(log_duration, true)
print(f"{'true level':<52}r = {r:>6.3f}  p = {p_text(p)}")
for (model_name, input_name), scores in expected_score.items():
    r, p = permutation_test(log_duration, scores)
    print(f"{f'{model_name}, {input_name}: expected score':<52}r = {r:>6.3f}  p = {p_text(p)}")

true level                                          r = -0.044  p = 0.570


Zero-shot, Video: expected score                    r =  0.478  p = <0.0001


Zero-shot, Video + commentary: expected score       r =  0.243  p = 0.001


Zero-shot, Commentary: expected score               r =  0.116  p = 0.128


Fine-tuned, Video: expected score                   r =  0.229  p = 0.002


Fine-tuned, Video + commentary: expected score      r =  0.117  p = 0.095


Fine-tuned, Commentary: expected score              r =  0.102  p = 0.161


In [5]:
import matplotlib.pyplot as plt

INPUT_STYLE = {  # color + marker, so the series stay distinguishable without color
    "Video": ("#2a78d6", "o"),
    "Video + commentary": ("#eb6834", "s"),
    "Commentary": ("#1baf7a", "^"),
}
REFERENCE_COLOR = "#8a8984"
x = np.arange(4)
tick_labels = [f"{label}\n{Counter(scenarios[quartile == q]).most_common(1)[0][0]}"
               for q, label in enumerate(quartile_labels)]
majority = [np.bincount(true[quartile == q]).max() / (quartile == q).sum() for q in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, model_name in zip(axes, ("Zero-shot", "Fine-tuned")):
    for input_name, (color, marker) in INPUT_STYLE.items():
        ok = correct[model_name, input_name]
        ax.plot(x, [ok[quartile == q].mean() for q in range(4)], color=color, lw=2, marker=marker, ms=7,
                markeredgecolor="white", markeredgewidth=1, label=input_name)
    ax.plot(x, majority, color=REFERENCE_COLOR, lw=1.5, ls="--", label="Majority class within bin")
    ax.set_title(model_name, loc="left", fontsize=11)
    ax.set_xticks(x, tick_labels, fontsize=9)
    ax.set_xlabel("Take duration quartile (most common scenario)")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("Accuracy")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=9, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f"Proficiency accuracy by take duration ({len(take_names)} test takes, ~{len(take_names) // 4} per quartile, "
             f"{NUM_FRAMES} frames per take)", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/accuracy_by_duration.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\darkn\AppData\Local\Temp\ipykernel_19548\4174722668.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
